# MainstreetAi — train the neural signal controller

This notebook trains the small neural network that times traffic lights in MainstreetAi's **Your city** view, tests it on cities it has never seen, and publishes it to Hugging Face.

**What gets trained:** an MLP (6 → 16 → 1, 129 parameters) shared across junctions. At each junction it reads the queues and nudges how the cycle is split between the two directions.

**How:** evolution strategies. The fitness of each candidate network is measured by running **the website's own traffic simulator** (`web/src/city/sim.js`) in Node, so the model is trained against exactly the code it will run in.

**Honest scoring:** the 37 ready-made cities are split into 22 for training, 5 for validation and 10 for test. The test cities are used once, at the end. The upload step refuses to publish a model that doesn't beat the current controller on those 10 cities, unless you override it.

Runtime: CPU is fine (no GPU needed). Use **Runtime → Run all**. Training takes about 30–40 minutes on a standard Colab machine.

## 1 · Get the code and check Node

In [ ]:
import os, subprocess, shutil
if not os.path.exists('MainstreetAi'):
    !git clone --depth 1 https://github.com/yefry08/MainstreetAi.git
%cd MainstreetAi

def node_major():
    try:
        return int(subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip().lstrip('v').split('.')[0])
    except Exception:
        return 0

# The simulator needs Node 18+. Colab usually has it; install only if not.
if node_major() < 18:
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
    !apt-get install -y nodejs > /dev/null
print('node', subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip())
print('ready-made cities:', len(os.listdir('web/public/cities')))

## 2 · Train

Each generation: 12 antithetic pairs of perturbed networks, each run on 6 random training cities for 5 simulated minutes. Every 5 generations the current network is scored on the validation cities. The generation with the best validation score is the one kept.

Colab sometimes disconnects long runs. Progress is saved at every validation, so if that happens just **re-run this cell** and it continues where it stopped.

`validation` is on the same scale as the rule-based reference printed at the start. 0 means no better than fixed timing; higher is better.

In [ ]:
WORKERS = os.cpu_count() or 2
GENERATIONS = 60
!python training/train.py train --generations {GENERATIONS} --workers {WORKERS} --resume


## 3 · Test on the 10 held-out cities

These cities have not been seen at any point. The protocol is 3 seeds per city and 15 simulated minutes per run. The network is compared against both fixed timing and the existing rule-based controller.

In [ ]:
!python training/train.py evaluate --workers {WORKERS}

In [ ]:
import json
model = json.load(open('training/out/model.json'))
summary = model['meta']['test']['summary']
print(json.dumps(summary, indent=2))

## 4 · Write the model card

In [ ]:
!python training/train.py card
from IPython.display import Markdown
Markdown(open('training/out/README.md').read().split('---', 2)[2])

## 5 · Publish to Hugging Face

1. Create a **write** token at https://huggingface.co/settings/tokens and paste it into the login box below. It stays in this Colab session; this notebook never prints or stores it.
2. Set `REPO_ID` to `your-username/mainstreetai-signal-controller`.

The upload is **refused** if the test above found the network no better than the current controller. Set `UPLOAD_ANYWAY = True` only if you want to publish it anyway (the model card will still report the result honestly).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi

REPO_ID = 'your-username/mainstreetai-signal-controller'   # <- change this
UPLOAD_ANYWAY = False

assert '/' in REPO_ID and not REPO_ID.startswith('your-username'), 'Set REPO_ID to your own username/repo first.'
if not summary['ship'] and not UPLOAD_ANYWAY:
    raise SystemExit('Not uploading: on the held-out cities this network did not beat the current controller. '
                     'Train longer or change the setup, or set UPLOAD_ANYWAY = True.')

api = HfApi()
api.create_repo(REPO_ID, repo_type='model', exist_ok=True)
for name in ('model.json', 'README.md'):
    api.upload_file(path_or_fileobj=f'training/out/{name}', path_in_repo=name,
                    repo_id=REPO_ID, commit_message='Upload MainstreetAi signal controller')
print(f'Published: https://huggingface.co/{REPO_ID}')
print(f'The site loads: https://huggingface.co/{REPO_ID}/resolve/main/model.json')

## 6 · Use it on the site

Send the `REPO_ID` back to whoever maintains the site. **Your city** will load the model from Hugging Face in each visitor's browser, with a copy shipped with the site as a fallback if Hugging Face can't be reached.